### This notebook shall be used to transform the raw DGCA data into a processed data with established hierarchy levels that will enable forecasting and reconciliation. The expected output is a long format dataset without any missing values and with three levels of hierarchy

In [ ]:
# Importing the necessary libraries
import pandas as pd
import numpy as np

In [ ]:
# Loading the data file
df = pd.read_csv('../data/raw/city_domestic.csv')

In [ ]:
df.shape

In [ ]:
# Adding a date column
df['date'] = pd.to_datetime(df[['Year', 'Month']].assign(DAY=1))

In [ ]:
print("Date Range: ",df['date'].min(), " to ",df['date'].max())

In [ ]:
# Directional Route Label
df['route'] = df['City1'] + " - " + df['City2']

In [ ]:
# Check for directional setup
sample = df[(df['City1'] == 'DELHI') & (df['City2'] == "MUMBAI")][['date','PaxToCity2','PaxFromCity2']].head()
print(sample)

In [ ]:
# Check for directional setup
sample = df[(df['City1'] == 'MUMBAI') & (df['City2'] == "DELHI")][['date','PaxToCity2','PaxFromCity2']].head()
print(sample)

In [ ]:
# Defining the target variable
df['y'] = df['PaxToCity2']

In [ ]:
# Aggregating the total volume and month count per route
route_stats = df.groupby('route').agg(
    total_volume = ('y','sum'),
    month_count = ('Month','count')
).reset_index()

In [ ]:
# Applying coverage filter: Selecting only those routes for which we have data for at least 60 months
route_stats_filtered = route_stats[route_stats['month_count']>=60]

In [ ]:
print(f"Routes after coverage filter (data available for minimum 60 months): {len(route_stats_filtered)}")

In [ ]:
# Applying volume filter: Selecting the top 200 routes by volume
top_routes = route_stats_filtered.nlargest(200,'total_volume')['route'].tolist()

In [ ]:
print(f"Final selected routes: {len(top_routes)}")

In [ ]:
selected_volume = route_stats_filtered[route_stats_filtered['route'].isin(top_routes)]['total_volume'].sum()
total_volume_sum = df['y'].sum()
print(f"200 selected routes cover {(selected_volume/total_volume_sum)*100:0.1f}% of directional volume")

In [ ]:
# Building the route level series
df_routes = df[df['route'].isin(top_routes)].copy() # Filtering the top routes

# Route level series (one row per route per month)
route_level = df_routes.groupby(['City1', 'City2', 'date'])['y'].sum().reset_index()

# Unique ID for the hierarchy
route_level['unique_id'] = 'TOTAL/' + route_level['City1'] + '/' + route_level['City1'] + '-' + route_level['City2']

# Dropping all unnecessary columns, renaming date to ds
route_level = route_level[['unique_id','date','y']].rename(columns = {'date':'ds'})

print("Route-level Series:",route_level['unique_id'].nunique(), " routes")
print(route_level.head())


In [ ]:
# Building the airport level series (sum of all outbound traffic from origin airport, per month)
airport_level = df_routes.groupby(['City1','date'])['y'].sum().reset_index()

# Unique ID for the hierarchy
airport_level['unique_id'] = 'TOTAL/' + airport_level['City1']

# Dropping all unnecessary columns and renaming date to ds
airport_level = airport_level[['unique_id','date','y']].rename(columns = {'date':'ds'})

print("Airport-level Series:",airport_level['unique_id'].nunique(), 'airports')
print(airport_level.head())

In [ ]:
# Building the national level series
total_level = df_routes.groupby(['date'])['y'].sum().reset_index()

# Unique ID for the hierarchy
total_level['unique_id'] = 'TOTAL'

# Dropping all unnecessary columns and renaming date as ds
total_level = total_level[['unique_id','date','y']].rename(columns = {'date':'ds'})

print("National level series:",total_level['unique_id'].nunique(), "series")
print(total_level.head())

print("Number of months:",len(total_level))

In [ ]:
# Stacking all three hierarchical levels together
hierarchy_df = pd.concat([total_level,airport_level,route_level], ignore_index = True)


# Sorting the df
hierarchy_df = hierarchy_df.sort_values(['unique_id','ds']).reset_index(drop = True)

print("Combined hierrachy dataset:")
print('Total rows: ',len(hierarchy_df))
print("Total unique series: ",hierarchy_df['unique_id'].nunique())

print("\nSeries count by level:")
print("TOTAL",(hierarchy_df['unique_id']=='TOTAL').sum() > 0)
print("Series Breakdown: ",hierarchy_df['unique_id'].str.count('/').value_counts().to_dict())

In [ ]:
# Handling missing values
def complete_series(group):
    # Set date index and resample to monthly start frequency
    g = group.set_index('ds').asfreq('MS')

    # Interpolate only internal gaps (limit_area = 'inside' doesn't fill before first or after last real value)
    g['y'] = g['y'].interpolate(method = 'linear', limit_area = 'inside')
    g['unique_id'] = group['unique_id'].iloc[0]
    return g.reset_index()

hierarchy_complete = hierarchy_df.groupby('unique_id',group_keys = False).apply(complete_series)

print("Before: ",len(hierarchy_df), " rows")
print("After: ",len(hierarchy_complete), " rows")

print("Remaining null values in y: ",hierarchy_complete['y'].isna().sum())

In [ ]:
# Coherence Check

check_month = '2025-12-01'

# Check 1: Do all airport series sum to TOTAL?
total_val = hierarchy_complete[
    (hierarchy_complete['unique_id'] == 'TOTAL') &
    (hierarchy_complete['ds'] == check_month)]['y'].values[0]

airport_sum = hierarchy_complete[
    (hierarchy_complete['unique_id'].str.count('/')==1) & 
    (hierarchy_complete['ds'] == check_month)]['y'].sum()

print(f"Check Month: {check_month}")
print(f"Total series value: {total_val}")
print(f"Sum of all airport series: {airport_sum}")
print(f"Match: {abs(total_val - airport_sum) < 1}")

In [ ]:
# Check 2: Do all route series sum to the origin airport?
check_month = '2025-12-01'
check_airport = 'MUMBAI'

# Airport level value
airport_val = hierarchy_complete[
    (hierarchy_complete['unique_id'] == f'TOTAL/{check_airport}') &
    (hierarchy_complete['ds'] == check_month)]['y'].values[0]

# Sum of all routes originating from the check airport
routes_sum = hierarchy_complete[
    (hierarchy_complete['unique_id'].str.startswith(f'TOTAL/{check_airport}/')) &
    (hierarchy_complete['ds'] == check_month)]['y'].sum()

print(f"Airport: {check_airport} & Month: {check_month}")
print(f"Airport series value: {airport_val}")
print(f"Sum of all route series: {routes_sum}")
print(f"Match: {abs(airport_val - routes_sum) < 1}")

In [ ]:
# Cleaning up and basic hygiene
hierarchy_complete = hierarchy_complete[['unique_id','ds','y']].sort_values(['unique_id','ds']).reset_index(drop = True)

# Saving the file
hierarchy_complete.to_csv('../data/processed/hierarchy_ready.csv',index = False)

print("Saved: ",hierarchy_complete.shape)

print(hierarchy_complete.head())

print("Unique series: ", hierarchy_complete['unique_id'].nunique())